# PyroLytic — Exploratory Data Analysis (v3)

**Status: dataset has grown from 68 to 146 rows since the last EDA pass** — a 115% increase, driven mainly by one 45-run primary-source LDPE study and several smaller targeted additions across PP/HDPE/mixed. This is the largest and most balanced version of the dataset to date. Still short of the 150-300 target (146/150), but functionally close enough to proceed to modeling.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
colors = {'PP': '#028090', 'HDPE': '#00A896', 'LDPE': '#02C39A', 'mixed': '#0B2545'}
df = pd.read_csv('pyrolysis_yields.csv')
df.shape

## 1. Dataset composition

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
counts = df['plastic_type'].value_counts()
bars = ax.bar(counts.index, counts.values, color=[colors.get(x, '#888') for x in counts.index])
ax.set_ylabel('Row count')
ax.set_title(f'Dataset Composition (n={len(df)})')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, str(int(bar.get_height())), ha='center', fontsize=11)
plt.tight_layout()
plt.show()

**Notable shift:** LDPE is now the largest category (59 rows, was 11 at the last checkpoint), overtaking PP. This came almost entirely from one primary-source 45-run Box-Behnken study — a reminder that this dataset's category balance is sensitive to whichever full-table papers get found, not a smooth natural distribution. Worth stating explicitly rather than implying organic balance.

## 2. Data confidence

In [ ]:
df['confidence'].value_counts().sort_index()

110/146 rows (75%) are now primary-source, exact values — up from 56% at the last checkpoint. Weighted-average confidence: 0.90 (was 0.82). This improved *despite* nearly doubling in size, because the strategy shifted toward targeting full experimental-design tables (which report every run as primary data) rather than scattered secondary citations.

## 3. Missing data check

In [ ]:
missing = df.isnull().sum()
missing[missing > 0]

**New gaps worth noting (didn't exist at the last checkpoint):**
- `temperature_C` now has 2 missing rows — two mixed-plastic heating-rate comparison points where only heating rate was reported, not temperature. These are unusable as full training rows without temperature and should be excluded or imputed carefully.
- `oil_yield_wt%` has 2 missing — two HDPE rows added specifically for their *gas* yield value (isothermal TGA data), included as partial data rather than left out entirely.
- `gas_yield_wt%` (73 missing) and `char_yield_wt%` (81 missing) remain the largest gaps — many sources only report oil/liquid yield. This directly limits multi-target modeling (Week 3-4 model card already flagged oil yield as the only target with full coverage; that's still true).
- `feedstock_mass_g` / `reactor_fill_pct` gained a few more populated rows but remain sparse (17/146) — still effectively usable only for the Papuga PP subset specifically.

## 4. LDPE Box-Behnken subset — heating rate effect (the dominant new signal)

In [ ]:
ldpe = df[(df['plastic_type']=='LDPE') & (df['source_doi']=='10.3390/recycling11030044')]
fig, ax = plt.subplots(figsize=(9, 5.5))
for hr, group in ldpe.groupby('heating_rate_C_min'):
    ax.scatter(group['temperature_C'], group['oil_yield_wt%'], label=f'{hr}°C/min', s=60, alpha=0.7)
ax.set_xlabel('Temperature (°C)')
ax.set_ylabel('Wax/Oil Yield (wt%)')
ax.set_title('LDPE Box-Behnken subset: Yield by Temperature, colored by Heating Rate (n=45)')
ax.legend(title='Heating Rate')
plt.tight_layout()
plt.show()

Slow heating (3°C/min, blue) visually clusters toward higher yields across both temperatures — consistent with the source paper's own ANOVA finding that heating rate was the dominant driver of wax yield, more influential than temperature itself. This is a genuinely large, clean, single-study subset (n=45) that could support its own narrow-scope model, similar in spirit to the Papuga PP subset from the Week 3-4 checkpoint — worth considering as a second narrow-scope validation case before attempting the full 146-row model again.

## 5. Oil yield vs. temperature, full dataset

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
for ptype, group in df.groupby('plastic_type'):
    valid = group.dropna(subset=['oil_yield_wt%'])
    ax.scatter(valid['temperature_C'], valid['oil_yield_wt%'], label=ptype,
               s=valid['confidence']*80, alpha=0.6, color=colors.get(ptype))
ax.set_xlabel('Temperature (°C)')
ax.set_ylabel('Oil Yield (wt%)')
ax.set_title(f'Oil Yield vs Temperature, All Data (n={df["oil_yield_wt%"].notna().sum()})')
ax.legend()
plt.tight_layout()
plt.show()

The classic peak-then-decline pyrolysis pattern remains visible at this larger scale, but the spread within any given temperature band is still wide — consistent with the Week 3-4 finding that temperature alone is not a strong predictor once reactor design, catalyst, and study-to-study differences are folded in. More data has NOT resolved that spread; it has made it more precisely visible. This is a hypothesis to re-test directly in the next modeling pass, not an assumption to carry over unchecked.

## 6. Summary — open questions for the next modeling pass

1. **Category balance shifted, not by design** — LDPE now dominates (59/146, 40%). Any model trained on the full dataset should be checked for whether it's implicitly becoming an "LDPE model with other categories as noise" rather than a genuinely general one.
2. **Confidence improved substantially** (0.82 → 0.90) — the dataset is more trustworthy on average, not just bigger.
3. **A second large, clean, single-study subset now exists** (LDPE Box-Behnken, n=45) — worth a narrow-scope model of its own, alongside the existing Papuga PP result, before re-attempting the full-dataset model.
4. **The original confound risk (Week 3-4: models leaning on 'which paper' rather than real chemistry) has not been re-tested at this larger scale.** This is the critical next step before any broad-scope R² number can be trusted — repeat the diagnostic procedure from the Week 3-4 checkpoint on the new 146-row dataset rather than assuming the fix generalizes.
5. **6 rows have real gaps** (2 missing temperature, 2 missing oil yield, both intentionally kept as partial data) — decide before training whether to drop, impute, or handle these explicitly.